# Compare Run Predictions Against Public Gold

This notebook compares predictions from `artifacts/runs/20260507T060219Z` against public demo gold files in `data/public/output/task_*/gold.csv`.

It reports two match types:

- `exact_match`: compares rows after trimming whitespace, preserving current column order and raw values.
- `rule_match`: approximates the rules-page behavior: ignores headers, ignores row order, tries column-order permutations, normalizes null-like values to empty strings, and rounds numeric-looking values to 2 decimals.

Gold files are only for local public-demo checking. Do not use them to generate official predictions.

In [15]:
from __future__ import annotations

import csv
import json
import math
import re
from collections import Counter
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
from itertools import permutations
from pathlib import Path

import pandas as pd
from IPython.display import display

RUN_ID = "20260515T221643Z"
PROJECT_ROOT = Path.cwd()
RUN_DIR = PROJECT_ROOT / "artifacts" / "runs" / RUN_ID
INPUT_DIR = PROJECT_ROOT / "data" / "public" / "input"
GOLD_DIR = PROJECT_ROOT / "data" / "public" / "output"

assert RUN_DIR.exists(), f"Missing run dir: {RUN_DIR}"
assert INPUT_DIR.exists(), f"Missing input dir: {INPUT_DIR}"
assert GOLD_DIR.exists(), f"Missing gold dir: {GOLD_DIR}"

print(f"Project root: {PROJECT_ROOT}")
print(f"Run dir:      {RUN_DIR}")

Project root: /Users/falcon/Machine Learning/KDDcup/starterkit
Run dir:      /Users/falcon/Machine Learning/KDDcup/starterkit/artifacts/runs/20260515T221643Z


In [16]:
NUMERIC_RE = re.compile(r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?$")
NULL_LIKE = {"", "null", "none", "nan", "na", "n/a"}


def task_sort_key(task_id: str) -> tuple[int, str]:
    try:
        return int(task_id.removeprefix("task_")), task_id
    except ValueError:
        return 10**9, task_id


def read_csv_table(path: Path) -> tuple[list[str], list[list[str]]]:
    if not path.exists():
        return [], []
    with path.open(newline="", encoding="utf-8-sig") as handle:
        rows = list(csv.reader(handle))
    if not rows:
        return [], []
    header = [str(value).strip() for value in rows[0]]
    body = [[str(value).strip() for value in row] for row in rows[1:]]
    return header, body


def normalize_cell(value: object) -> str:
    text = "" if value is None else str(value).strip()
    if text.lower() in NULL_LIKE:
        return ""
    if NUMERIC_RE.match(text):
        try:
            rounded = Decimal(text).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
            return format(rounded, "f")
        except InvalidOperation:
            return text
    return text


def normalize_rows(rows: list[list[str]]) -> list[tuple[str, ...]]:
    return [tuple(normalize_cell(value) for value in row) for row in rows]


def stripped_rows(rows: list[list[str]]) -> list[tuple[str, ...]]:
    return [tuple(str(value).strip() for value in row) for row in rows]


def row_counter(rows: list[tuple[str, ...]]) -> Counter[tuple[str, ...]]:
    return Counter(rows)


def column_count(header: list[str], rows: list[list[str]]) -> int:
    if header:
        return len(header)
    if rows:
        return len(rows[0])
    return 0


def widths_are_consistent(rows: list[list[str]], expected_width: int) -> bool:
    return all(len(row) == expected_width for row in rows)


def compare_exact(pred_rows: list[list[str]], gold_rows: list[list[str]]) -> dict[str, object]:
    pred_counter = row_counter(stripped_rows(pred_rows))
    gold_counter = row_counter(stripped_rows(gold_rows))
    missing_counter = gold_counter - pred_counter
    extra_counter = pred_counter - gold_counter
    return {
        "ok": pred_counter == gold_counter,
        "missing_count": sum(missing_counter.values()),
        "extra_count": sum(extra_counter.values()),
        "missing_sample": list(missing_counter.elements())[:5],
        "extra_sample": list(extra_counter.elements())[:5],
    }


def compare_rule_style(
    pred_header: list[str],
    pred_rows: list[list[str]],
    gold_header: list[str],
    gold_rows: list[list[str]],
    max_permutations: int = 40320,
) -> dict[str, object]:
    pred_width = column_count(pred_header, pred_rows)
    gold_width = column_count(gold_header, gold_rows)
    if pred_width != gold_width:
        return {
            "ok": False,
            "reason": f"column count differs: pred={pred_width}, gold={gold_width}",
            "best_perm": None,
            "missing_count": len(gold_rows),
            "extra_count": len(pred_rows),
            "missing_sample": normalize_rows(gold_rows)[:5],
            "extra_sample": normalize_rows(pred_rows)[:5],
        }
    if not widths_are_consistent(pred_rows, pred_width):
        return {"ok": False, "reason": "prediction row widths are inconsistent", "best_perm": None}
    if not widths_are_consistent(gold_rows, gold_width):
        return {"ok": False, "reason": "gold row widths are inconsistent", "best_perm": None}

    normalized_gold = normalize_rows(gold_rows)
    gold_counter = row_counter(normalized_gold)
    normalized_pred = normalize_rows(pred_rows)

    if pred_width <= 1:
        candidate_perms = [tuple(range(pred_width))]
    elif math.factorial(pred_width) <= max_permutations:
        candidate_perms = list(permutations(range(pred_width)))
    else:
        candidate_perms = [tuple(range(pred_width))]

    best = None
    for perm in candidate_perms:
        permuted_pred = [tuple(row[index] for index in perm) for row in normalized_pred]
        pred_counter = row_counter(permuted_pred)
        overlap = sum((pred_counter & gold_counter).values())
        missing_counter = gold_counter - pred_counter
        extra_counter = pred_counter - gold_counter
        candidate = {
            "ok": pred_counter == gold_counter,
            "reason": "",
            "best_perm": perm,
            "overlap": overlap,
            "missing_count": sum(missing_counter.values()),
            "extra_count": sum(extra_counter.values()),
            "missing_sample": list(missing_counter.elements())[:5],
            "extra_sample": list(extra_counter.elements())[:5],
        }
        if candidate["ok"]:
            return candidate
        if best is None or candidate["overlap"] > best["overlap"]:
            best = candidate

    return best or {
        "ok": False,
        "reason": "no comparable rows",
        "best_perm": None,
        "overlap": 0,
        "missing_count": len(gold_rows),
        "extra_count": len(pred_rows),
        "missing_sample": normalize_rows(gold_rows)[:5],
        "extra_sample": normalize_rows(pred_rows)[:5],
    }

In [17]:
summary_path = RUN_DIR / "summary.json"
summary = json.loads(summary_path.read_text()) if summary_path.exists() else {"tasks": []}
summary_by_task = {task["task_id"]: task for task in summary.get("tasks", [])}

records = []
task_json_paths = sorted(INPUT_DIR.glob("task_*/task.json"), key=lambda p: task_sort_key(p.parent.name))

for task_json_path in task_json_paths:
    task = json.loads(task_json_path.read_text())
    task_id = task["task_id"]
    pred_path = RUN_DIR / task_id / "prediction.csv"
    gold_path = GOLD_DIR / task_id / "gold.csv"

    pred_header, pred_rows = read_csv_table(pred_path)
    gold_header, gold_rows = read_csv_table(gold_path)
    exact = compare_exact(pred_rows, gold_rows) if pred_path.exists() and gold_path.exists() else {"ok": False}
    rule = (
        compare_rule_style(pred_header, pred_rows, gold_header, gold_rows)
        if pred_path.exists() and gold_path.exists()
        else {"ok": False, "reason": "missing prediction or gold"}
    )
    run_task = summary_by_task.get(task_id, {})

    records.append(
        {
            "task_id": task_id,
            "difficulty": task.get("difficulty"),
            "question": task.get("question"),
            "run_succeeded": bool(run_task.get("succeeded", pred_path.exists())),
            "failure_reason": run_task.get("failure_reason"),
            "has_prediction": pred_path.exists(),
            "gold_rows": len(gold_rows),
            "pred_rows": len(pred_rows),
            "gold_cols": column_count(gold_header, gold_rows),
            "pred_cols": column_count(pred_header, pred_rows),
            "exact_match": bool(exact.get("ok")),
            "rule_match": bool(rule.get("ok")),
            "rule_reason": rule.get("reason", ""),
            "missing_count": rule.get("missing_count"),
            "extra_count": rule.get("extra_count"),
            "missing_sample": rule.get("missing_sample"),
            "extra_sample": rule.get("extra_sample"),
            "prediction_path": str(pred_path),
            "gold_path": str(gold_path),
        }
    )

comparison_df = pd.DataFrame(records)
comparison_df.head()

,task_id,difficulty,question,run_succeeded,failure_reason,has_prediction,gold_rows,pred_rows,gold_cols,pred_cols,exact_match,rule_match,rule_reason,missing_count,extra_count,missing_sample,extra_sample,prediction_path,gold_path
0,task_11,easy,"For patients with severe degree of thrombosis,...",True,None,True,3,3,3,3,False,False,,2.0,2.0,"[(163109.00, F, SLE), (2803470.00, F, SLE)]","[(163109.00, F, ), (2803470.00, F, SLE+Psy)]",/Users/falcon/Machine Learning/KDDcup/starterk...,/Users/falcon/Machine Learning/KDDcup/starterk...
1,task_19,easy,List the full name of the Student_Club members...,True,None,True,3,3,2,2,True,True,,0.0,0.0,[],[],/Users/falcon/Machine Learning/KDDcup/starterk...,/Users/falcon/Machine Learning/KDDcup/starterk...
2,task_22,easy,State the date Connor Hilton paid his/her dues.,True,None,True,2,2,1,1,True,True,,0.0,0.0,[],[],/Users/falcon/Machine Learning/KDDcup/starterk...,/Users/falcon/Machine Learning/KDDcup/starterk...
3,task_24,easy,"How many members attended the ""Women's Soccer""...",True,None,True,1,1,1,1,True,True,,0.0,0.0,[],[],/Users/falcon/Machine Learning/KDDcup/starterk...,/Users/falcon/Machine Learning/KDDcup/starterk...
4,task_25,easy,Which event has the lowest cost?,True,None,True,3,3,1,1,True,True,,0.0,0.0,[],[],/Users/falcon/Machine Learning/KDDcup/starterk...,/Users/falcon/Machine Learning/KDDcup/starterk...


In [18]:
summary_cols = [
    "task_id",
    "difficulty",
    "run_succeeded",
    "has_prediction",
    "exact_match",
    "rule_match",
    "gold_rows",
    "pred_rows",
    "gold_cols",
    "pred_cols",
    "missing_count",
    "extra_count",
]

print(f"Tasks: {len(comparison_df)}")
print(f"Predictions present: {comparison_df['has_prediction'].sum()}")
print(f"Exact matches: {comparison_df['exact_match'].sum()}")
print(f"Rule-style matches: {comparison_df['rule_match'].sum()}")

display(
    comparison_df.groupby("difficulty", dropna=False)
    .agg(
        tasks=("task_id", "count"),
        predictions=("has_prediction", "sum"),
        exact_matches=("exact_match", "sum"),
        rule_matches=("rule_match", "sum"),
    )
    .reset_index()
)

display(comparison_df[summary_cols].sort_values("task_id", key=lambda s: s.map(task_sort_key)))

Tasks: 50
Predictions present: 19
Exact matches: 17
Rule-style matches: 18


,difficulty,tasks,predictions,exact_matches,rule_matches
0,easy,15,15,14,14
1,extreme,1,0,0,0
2,hard,11,0,0,0
3,medium,23,4,3,4


,task_id,difficulty,run_succeeded,has_prediction,exact_match,rule_match,gold_rows,pred_rows,gold_cols,pred_cols,missing_count,extra_count
0,task_11,easy,True,True,False,False,3,3,3,3,2.0,2.0
1,task_19,easy,True,True,True,True,3,3,2,2,0.0,0.0
2,task_22,easy,True,True,True,True,2,2,1,1,0.0,0.0
3,task_24,easy,True,True,True,True,1,1,1,1,0.0,0.0
4,task_25,easy,True,True,True,True,3,3,1,1,0.0,0.0
5,task_26,easy,True,True,True,True,1,1,1,1,0.0,0.0
6,task_27,easy,True,True,True,True,1,1,3,3,0.0,0.0
7,task_38,easy,True,True,True,True,140,140,1,1,0.0,0.0
8,task_64,easy,True,True,True,True,4,4,1,1,0.0,0.0
9,task_67,easy,True,True,True,True,1,1,1,1,0.0,0.0


In [19]:
mismatches = comparison_df[~comparison_df["rule_match"]].copy()
display(
    mismatches[
        [
            "task_id",
            "difficulty",
            "run_succeeded",
            "has_prediction",
            "rule_reason",
            "missing_count",
            "extra_count",
            "missing_sample",
            "extra_sample",
            "question",
        ]
    ].sort_values("task_id", key=lambda s: s.map(task_sort_key))
)

,task_id,difficulty,run_succeeded,has_prediction,rule_reason,missing_count,extra_count,missing_sample,extra_sample,question
0,task_11,easy,True,True,,2.0,2.0,"[(163109.00, F, SLE), (2803470.00, F, SLE)]","[(163109.00, F, ), (2803470.00, F, SLE+Psy)]","For patients with severe degree of thrombosis,..."
16,task_163,medium,False,False,missing prediction or gold,NaN,NaN,None,None,Identify the type of expenses and their total ...
20,task_194,medium,False,False,missing prediction or gold,NaN,NaN,None,None,What are the bonds that have phosphorus and ni...
21,task_196,medium,False,False,missing prediction or gold,NaN,NaN,None,None,What is the average number of bonds the atoms ...
22,task_199,medium,False,False,missing prediction or gold,NaN,NaN,None,None,List the names and funding types of schools fr...
23,task_200,medium,False,False,missing prediction or gold,NaN,NaN,None,None,Calculate the total atoms with triple-bond mol...
24,task_214,medium,False,False,missing prediction or gold,NaN,NaN,None,None,How many Brazilian Portuguese translated sets ...
25,task_218,medium,False,False,missing prediction or gold,NaN,NaN,None,None,What is the telephone number for the school wi...
26,task_243,medium,False,False,missing prediction or gold,NaN,NaN,None,None,"For the user No.24, how many times is the numb..."
27,task_249,medium,False,False,missing prediction or gold,NaN,NaN,None,None,What is the average of the up votes and the av...


In [20]:
# Change TASK_ID to inspect a specific mismatch side by side.
TASK_ID = "task_22"

pred_path = RUN_DIR / TASK_ID / "prediction.csv"
gold_path = GOLD_DIR / TASK_ID / "gold.csv"
pred_header, pred_rows = read_csv_table(pred_path)
gold_header, gold_rows = read_csv_table(gold_path)

print("Prediction:", pred_path)
display(pd.DataFrame(pred_rows, columns=pred_header if pred_header else None))

print("Gold:", gold_path)
display(pd.DataFrame(gold_rows, columns=gold_header if gold_header else None))

print("Exact:", compare_exact(pred_rows, gold_rows))
print("Rule-style:", compare_rule_style(pred_header, pred_rows, gold_header, gold_rows))

Prediction: /Users/falcon/Machine Learning/KDDcup/starterkit/artifacts/runs/20260515T221643Z/task_22/prediction.csv


,date_received
0,2019-09-12
1,2019-10-02


Gold: /Users/falcon/Machine Learning/KDDcup/starterkit/data/public/output/task_22/gold.csv


,date_received
0,2019-10-02
1,2019-09-12


Exact: {'ok': True, 'missing_count': 0, 'extra_count': 0, 'missing_sample': [], 'extra_sample': []}
Rule-style: {'ok': True, 'reason': '', 'best_perm': (0,), 'overlap': 2, 'missing_count': 0, 'extra_count': 0, 'missing_sample': [], 'extra_sample': []}


In [21]:
# Optional: export the comparison table next to the run artifacts.
report_path = RUN_DIR / "comparison_report.csv"
comparison_df.to_csv(report_path, index=False)
print(f"Wrote {report_path}")

Wrote /Users/falcon/Machine Learning/KDDcup/starterkit/artifacts/runs/20260515T221643Z/comparison_report.csv
